In [1]:
import pandas as pd

In [2]:
file_path = "C:/Users/happy/Downloads/marketing_campaign_response_dataset.csv"
# Read the CSV file and load it into a pandas DataFrame
df = pd.read_csv(file_path)

In [3]:
print(df.head())
print(df.shape)
print(df.columns)
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df.duplicated().sum())

   Customer_ID   Age  Gender     City   Income Customer_Segment  \
0            1  56.0    Male      NaN  99172.0              VIP   
1            2  69.0  Female  Vaughan  40916.0           Budget   
2            3  46.0  Female  Markham  34809.0           Budget   
3            4  32.0  Female   Oshawa  62201.0          Premium   
4            5  60.0  Female      NaN  83330.0         Standard   

   Website_Visits Email_Opened  Ad_Clicks  Previous_Purchases  \
0               6           No          9                   2   
1               8          Yes         10                   4   
2               7          Yes          4                   8   
3              22          Yes          3                  11   
4              28          Yes          1                   2   

  Discount_Offered Campaign_Channel  Customer_Satisfaction  \
0              Yes              SMS                    1.0   
1              Yes     Social Media                    4.0   
2              Yes  

In [4]:
print("Number of duplicate rows before cleaning:", df.duplicated().sum())
df = df.drop_duplicates()
print("Number of duplicate rows after cleaning:", df.duplicated().sum())


Number of duplicate rows before cleaning: 5
Number of duplicate rows after cleaning: 0


In [5]:
# Check missing values
print(df.isnull().sum())
# Fill missing numerical values with the median
# Median is often safer than mean when the column has extreme values.
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Income"] = df["Income"].fillna(df["Income"].median())
df["Customer_Satisfaction"] = df["Customer_Satisfaction"].fillna(df["Customer_Satisfaction"].median())
df["Gender"] = df["Gender"].fillna("Unknown")
df["City"] = df["City"].fillna("Unknown")
df["Customer_Segment"] = df["Customer_Segment"].fillna("Unknown")

Customer_ID              0
Age                      5
Gender                   5
City                     5
Income                   5
Customer_Segment         5
Website_Visits           0
Email_Opened             0
Ad_Clicks                0
Previous_Purchases       0
Discount_Offered         0
Campaign_Channel         0
Customer_Satisfaction    5
Last_Month_Spending      0
Responded                0
dtype: int64


In [6]:
text_columns = ['Customer_ID', 'Age', 'Gender', 'City', 'Income', 'Customer_Segment',
       'Website_Visits', 'Email_Opened', 'Ad_Clicks', 'Previous_Purchases',
       'Discount_Offered', 'Campaign_Channel', 'Customer_Satisfaction',
       'Last_Month_Spending', 'Responded']
for col in text_columns:
    df[col] = df[col].astype(str).str.strip()
# Example: make city names consistent
# df["City"] = df["City"].str.title()

In [7]:
feature_columns = [
    "Age",
    "Gender",
    "City",
    "Income",
    "Customer_Segment",
    "Website_Visits",
    "Email_Opened",
    "Ad_Clicks",
    "Previous_Purchases",
    "Discount_Offered",
    "Campaign_Channel",
    "Customer_Satisfaction",
    "Last_Month_Spending"
]

X = df[feature_columns]
y = df["Responded"].str.strip().map({"No": 0, "Yes": 1})
print(X.head())
print(y.head())

    Age  Gender     City   Income Customer_Segment Website_Visits  \
0  56.0    Male  Unknown  99172.0              VIP              6   
1  69.0  Female  Vaughan  40916.0           Budget              8   
2  46.0  Female  Markham  34809.0           Budget              7   
3  32.0  Female   Oshawa  62201.0          Premium             22   
4  60.0  Female  Unknown  83330.0         Standard             28   

  Email_Opened Ad_Clicks Previous_Purchases Discount_Offered Campaign_Channel  \
0           No         9                  2              Yes              SMS   
1          Yes        10                  4              Yes     Social Media   
2          Yes         4                  8              Yes            Email   
3          Yes         3                 11              Yes            Email   
4          Yes         1                  2              Yes              SMS   

  Customer_Satisfaction Last_Month_Spending  
0                   1.0                 389  
1     

In [8]:
# Step 5: Split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.20,
random_state=42,
stratify=y
)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (240, 13)
X_test shape: (60, 13)
y_train shape: (240,)
y_test shape: (60,)


In [9]:
# Step 6: Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numerical_features = [
    "Age",
    "Income",
    "Website_Visits",
    "Ad_Clicks",
    "Previous_Purchases",
    "Customer_Satisfaction",
    "Last_Month_Spending"
]

categorical_features = [
    "Gender",
    "City",
    "Customer_Segment",
    "Email_Opened",
    "Discount_Offered",
    "Campaign_Channel"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)


Processed training shape: (240, 30)
Processed test shape: (60, 30)


In [10]:
# Step 7: Train a baseline model
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000)

baseline_model.fit(X_train_processed, y_train)

y_pred = baseline_model.predict(X_test_processed)

print(y_pred)

[1 0 0 0 1 1 1 0 1 0 1 1 0 0 1 1 1 0 1 1 0 1 1 0 1 1 0 1 0 0 0 0 1 0 0 1 0
 0 1 1 0 1 1 1 1 0 1 0 1 0 0 0 0 0 1 1 1 1 1 1]


In [11]:
# Cleaner workflow: preprocessing + model in one Pipeline
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
clf = Pipeline(steps=[
("preprocessor", preprocessor),
("model", LogisticRegression(max_iter=1000))
])
# Train the full pipeline on raw X_train
clf.fit(X_train, y_train)
# Predict using raw X_test; preprocessing is applied automatically
y_pred = clf.predict(X_test)
print(y_pred)


[1 0 0 0 1 1 1 0 1 0 1 1 0 0 1 1 1 0 1 1 0 1 1 0 1 1 0 1 0 0 0 0 1 0 0 1 0
 0 1 1 0 1 1 1 1 0 1 0 1 0 0 0 0 0 1 1 1 1 1 1]


In [12]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7833333333333333
Confusion Matrix:
[[22  8]
 [ 5 25]]
Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.73      0.77        30
           1       0.76      0.83      0.79        30

    accuracy                           0.78        60
   macro avg       0.79      0.78      0.78        60
weighted avg       0.79      0.78      0.78        60

